# Task 3: Broadcast Join Experiment for Query 2



## Connect to Spark

In [1]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query
from analytics.source_tables import register_broadcast_join_sources

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-BroadcastJoin-Query3')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:05:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.catalog.clearCache()

## Isolate the join strategy

- Automatic broadcasting is disabled
- AQE is also disabled so it cannot change the join at [runtime](https://spark.apache.org/docs/4.0.4/sql-performance-tuning.html#adaptive-query-execution).
- The forced query uses a broadcast hint.

In [3]:
JOIN_CONFIGS = {
    'spark.sql.adaptive.enabled': 'false',
    'spark.sql.autoBroadcastJoinThreshold': '-1',
}
for key, value in JOIN_CONFIGS.items():
    spark.conf.set(key, value)

JOIN_CONFIGS

{'spark.sql.adaptive.enabled': 'false',
 'spark.sql.autoBroadcastJoinThreshold': '-1'}

## Measure normal and broadcast joins

In [4]:
NORMAL_QUERY = 'query_02'
BROADCAST_QUERY = 'query_02_bj'
REPETITIONS = 3

normal_results = measure_registered_query(spark, NORMAL_QUERY, REPETITIONS)
broadcast_results = measure_registered_query(spark, BROADCAST_QUERY, REPETITIONS)

assert rows_match(
    normal_results['rows'],
    broadcast_results['rows'],
), 'Normal and broadcast results differ.'

print('Normal and broadcast results match within float tolerance.')

26/09/19 10:05:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/19 10:05:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/19 10:06:12 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:06:17 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:06:21 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:06:26 WARN HintErrorLogger: Count not find relation 'air_quality' specified in h

Normal and broadcast results match within float tolerance.


## Normal-join physical plan

This indicates normal join was used: `SortMergeJoin`.

In [5]:
execute_registered_query(spark, NORMAL_QUERY).explain('formatted')

== Physical Plan ==
* Sort (18)
+- Exchange (17)
   +- * HashAggregate (16)
      +- Exchange (15)
         +- * HashAggregate (14)
            +- * Project (13)
               +- * SortMergeJoin Inner (12)
                  :- * Sort (6)
                  :  +- Exchange (5)
                  :     +- * Project (4)
                  :        +- * Filter (3)
                  :           +- * ColumnarToRow (2)
                  :              +- Scan parquet  (1)
                  +- * Sort (11)
                     +- Exchange (10)
                        +- * Filter (9)
                           +- * ColumnarToRow (8)
                              +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [trip_distance#2031, coco_pu#2070, date#2085]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,coco_pu:int>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [trip_dis

## Broadcast-join physical plan

- These indicate broadcast join was used: `BroadcastHashJoin` and `BroadcastExchange`.

In [6]:
execute_registered_query(spark, BROADCAST_QUERY).explain('formatted')

26/09/19 10:07:03 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.


== Physical Plan ==
* Sort (18)
+- Exchange (17)
   +- * HashAggregate (16)
      +- Exchange (15)
         +- * HashAggregate (14)
            +- * Project (13)
               +- * SortMergeJoin Inner (12)
                  :- * Sort (6)
                  :  +- Exchange (5)
                  :     +- * Project (4)
                  :        +- * Filter (3)
                  :           +- * ColumnarToRow (2)
                  :              +- Scan parquet  (1)
                  +- * Sort (11)
                     +- Exchange (10)
                        +- * Filter (9)
                           +- * ColumnarToRow (8)
                              +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [trip_distance#2238, coco_pu#2277, date#2292]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,coco_pu:int>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [trip_dis

## Benchmark summary

In [7]:
summary = {
    'normal_samples_seconds': [round(value, 3) for value in normal_results['samples_seconds']],
    'normal_median_seconds': round(normal_results['median_seconds'], 3),
    'broadcast_samples_seconds': [round(value, 3) for value in broadcast_results['samples_seconds']],
    'broadcast_median_seconds': round(broadcast_results['median_seconds'], 3),
    'broadcast_speedup_factor': round(normal_results['median_seconds'] / broadcast_results['median_seconds'],2),
}
summary

{'normal_samples_seconds': [6.783, 5.579, 4.523],
 'normal_median_seconds': 5.579,
 'broadcast_samples_seconds': [3.954, 4.259, 3.977],
 'broadcast_median_seconds': 3.977,
 'broadcast_speedup_factor': 1.4}

## Cleanup

In [8]:
spark.catalog.clearCache()
spark.stop()